# LLM-as-a-Judge Validation - ClearView Synthetic Data

## Purpose
This notebook validates the quality of LLM-generated synthetic cosmetic reviews used to address class imbalance in the ClearView MAMSR training corpus.

```
Input : data/augmented/LLM_Gen_*.csv   (synthetic reviews per aspect/sentiment)
Output: data/augmented/validation_output/validation_results.csv
        data/augmented/validation_output/validation_report.md
```

## Methodology
Follows the **Multi-Model LLM-as-a-Judge** protocol to avoid single-model bias:

- **3 locally hosted models** (via [Ollama](https://ollama.com)) are consulted independently:
  - `llama3.1` - Meta's Llama 3.1 8B
  - `mistral` - Mistral 7B
  - `gemma2` - Google Gemma 2 9B
- Each model is instantiated as **3 independent judge personas**:
  - **Judge A** - Strict NLP annotation expert
  - **Judge B** - Senior data quality assessor
  - **Judge C** - Consumer research analyst (lenient)
- This yields **9 independent judges** per review (3 models × 3 personas)
- Each review is scored on **3 criteria**:
  | Criterion | Description |
  |-----------|-------------|
  | `label_correctness` | Does the review express the claimed sentiment class? |
  | `aspect_relevance` | Does the review discuss the claimed cosmetic aspect? |
  | `naturalness` | Does it read like a genuine customer review? |
- **Per-model majority vote** (≥ 2 of 3 personas) determines each model's verdict per criterion.
- **Cross-model average** (≥ 2 of 3 models pass) determines the final verdict per criterion.
- **Fleiss' Kappa (κ)** across all 9 raters quantifies inter-rater reliability.

## Files Validated
| File | Aspect | Sentiment |
|------|--------|-----------|
| `LLM_Gen_Packing_Neg_Reviews.csv` | packing | negative |
| `LLM_Gen_Packing_Neu_Reviews.csv` | packing | neutral |
| `LLM_Gen_Price_Neg_Reviews.csv` | price | negative |
| `LLM_Gen_Price_Neu_Reviews.csv` | price | neutral |
| `LLM_Gen_Smell_Neu_Reviews.csv` | smell | neutral |

## Prerequisites
```bash
pip install requests pandas numpy tqdm
ollama pull llama3.1
ollama pull mistral
ollama pull gemma2
```
> **Note:** Ollama must be running locally at `http://localhost:11434` before executing this notebook.

## 1. Imports & Settings

All standard library and third-party imports, plus the two core configuration blocks:
- **Ollama settings** - base URL and list of judge models
- **Domain config** - which CSV files to validate, aspect descriptions, sentiment descriptions, and judge criteria

In [1]:
print("Starting: Loading imports and configuration...")
import time
_start_time = time.time()

import json
import random
import re
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

# ─── Ollama Settings ──────────────────────────────────────────────────────────
OLLAMA_BASE = "http://localhost:11434"

# Multiple judge models — each model evaluates every review independently
# to avoid inheriting the bias of a single model.
JUDGE_MODELS = [
    "llama3.1",   # Meta Llama 3.1 8B
    "mistral",    # Mistral 7B
    "gemma2",     # Google Gemma 2 9B
]

# ─── Runtime Config ───────────────────────────────────────────────────────────
DATA_DIR   = Path("../data/augmented")
OUTPUT_DIR = Path("../data/augmented/validation_output")
SAMPLE_N   = 30   # reviews sampled per CSV file

print(f"  Data dir    : {DATA_DIR.resolve()}")
print(f"  Output dir  : {OUTPUT_DIR.resolve()}")
print(f"  Sample N    : {SAMPLE_N} per file")
print(f"  Judge models: {JUDGE_MODELS}")
print(f"  Total judges: {len(JUDGE_MODELS)} models × 3 personas = {len(JUDGE_MODELS)*3} per review")

print(f"\n Done in {time.time() - _start_time:.2f}s")
print("Completed: Loading imports and configuration.")

Starting: Loading imports and configuration...
  Data dir    : C:\Users\lucif\Desktop\Clearview\ml-research\data\augmented
  Output dir  : C:\Users\lucif\Desktop\Clearview\ml-research\data\augmented\validation_output
  Sample N    : 30 per file
  Judge models: ['llama3.1', 'mistral', 'gemma2']
  Total judges: 3 models × 3 personas = 9 per review

Done in 1.07s
Completed: Loading imports and configuration.


## 2. Domain Configuration

Defines:
- **`SYNTHETIC_FILES`** - maps each CSV filename to its `(aspect, sentiment)` ground-truth labels
- **`ASPECT_DESCRIPTIONS`** - plain-English descriptions of each aspect, injected into judge prompts
- **`SENTIMENT_DESCRIPTIONS`** - describes what each sentiment class should express
- **`CRITERIA`** - the three quality dimensions scored by every judge

In [2]:
print("Starting: Setting domain configuration...")
_start_time = time.time()

SYNTHETIC_FILES = {
    "LLM_Gen_Packing_Neg_Reviews.csv":  ("packing",  "negative"),
    "LLM_Gen_Packing_Neu_Reviews.csv":  ("packing",  "neutral"),
    "LLM_Gen_Price_Neg_Reviews.csv":    ("price",    "negative"),
    "LLM_Gen_Price_Neu_Reviews.csv":    ("price",    "neutral"),
    "LLM_Gen_Smell_Neu_Reviews.csv":    ("smell",    "neutral"),
}

ASPECT_DESCRIPTIONS = {
    "packing":      "packaging, box, bottle, container, seal, wrapper",
    "price":        "cost, price, value for money, affordability",
    "smell":        "scent, fragrance, odour, aroma of the product",
    "texture":      "feel, consistency, thickness, spreadability on skin",
    "colour":       "colour, shade, pigmentation, hue",
    "shipping":     "delivery, courier, arrival condition",
    "stayingpower": "how long the product lasts, wear time, longevity",
}

SENTIMENT_DESCRIPTIONS = {
    "positive": "satisfaction, praise, or approval",
    "neutral":  "factual, balanced, or neither clearly positive nor negative",
    "negative": "dissatisfaction, complaint, or criticism",
}

CRITERIA = ["label_correctness", "aspect_relevance", "naturalness"]

print(f"  Files to validate : {len(SYNTHETIC_FILES)}")
print(f"  Aspects covered   : {sorted(set(v[0] for v in SYNTHETIC_FILES.values()))}")
print(f"  Criteria          : {CRITERIA}")

print(f"\n Done in {time.time() - _start_time:.2f}s")
print("Completed: Setting domain configuration.")

Starting: Setting domain configuration...
  Files to validate : 5
  Aspects covered   : ['packing', 'price', 'smell']
  Criteria          : ['label_correctness', 'aspect_relevance', 'naturalness']

Done in 0.00s
Completed: Setting domain configuration.


## 3. Judge Personas & Prompt Template

Three separate system prompts instantiate different annotation personas within **each** model:

| Persona | Role | Bias |
|---------|------|------|
| **Judge A** | Strict NLP annotation expert | Marks ambiguous reviews as FAIL |
| **Judge B** | Senior data quality assessor | Applies moderate, consistent standards |
| **Judge C** | Consumer research analyst | Gives benefit of the doubt |

Each model runs all 3 personas → **9 total judge calls per review**.  
Verdict pipeline: 3 personas → per-model majority vote → average across models → final verdict.

In [3]:
print(" Starting: Defining judge personas and prompt template...")
_start_time = time.time()

JUDGE_PERSONAS = [
    {
        "name": "Judge_A",
        "system": (
            "You are a strict NLP annotation expert specialising in Aspect-Based Sentiment Analysis. "
            "You evaluate synthetic cosmetic reviews with high standards. "
            "If a review is ambiguous, mark it FAIL. "
            "You MUST respond ONLY with a valid JSON object. No explanation, no extra text."
        ),
    },
    {
        "name": "Judge_B",
        "system": (
            "You are a senior data quality assessor for an NLP research team. "
            "Validate whether machine-generated cosmetic reviews are suitable for training a sentiment classifier. "
            "Apply consistent, moderate standards. "
            "You MUST respond ONLY with a valid JSON object. No explanation, no extra text."
        ),
    },
    {
        "name": "Judge_C",
        "system": (
            "You are a consumer research analyst experienced with cosmetic product reviews. "
            "Assess whether auto-generated reviews are realistic and correctly labelled. "
            "Give benefit of the doubt when intent is clear. "
            "You MUST respond ONLY with a valid JSON object. No explanation, no extra text."
        ),
    },
]

JUDGE_PROMPT = """Evaluate this synthetic cosmetic product review.

REVIEW TEXT:
\"\"\"{review_text}\"\"\"

CLAIMED ASPECT: {aspect}
(Review should discuss: {aspect_description})

CLAIMED SENTIMENT: {sentiment}
(Review should express: {sentiment_description})

Evaluate on exactly three criteria. Respond ONLY with this JSON, nothing else:

{{
  "label_correctness": {{"verdict": "PASS", "reason": "one sentence why"}},
  "aspect_relevance":  {{"verdict": "PASS", "reason": "one sentence why"}},
  "naturalness":       {{"verdict": "PASS", "reason": "one sentence why"}}
}}

Rules:
- label_correctness = PASS if review clearly expresses {sentiment} sentiment toward {aspect}
- aspect_relevance  = PASS if review actually discusses {aspect} content
- naturalness       = PASS if review reads like a genuine human-written cosmetic review
- Use FAIL if a criterion is not met
- Output ONLY the JSON object, no other text before or after
"""

print(f"  Judges defined : {[j['name'] for j in JUDGE_PERSONAS]}")

print(f"\n Done in {time.time() - _start_time:.2f}s")
print("Completed: Defining judge personas and prompt template.")

Starting: Defining judge personas and prompt template...
  Judges defined : ['Judge_A', 'Judge_B', 'Judge_C']

Done in 0.00s
Completed: Defining judge personas and prompt template.


## 4. Ollama API Functions (with GPU Detection)

- **`check_ollama(model)`** - verifies that Ollama is running and the requested model is downloaded; reports GPU/CPU status.
- **`check_all_models()`** - checks every model in `JUDGE_MODELS` and returns the resolved model tags.
- **`call_ollama(system_prompt, user_prompt, model)`** - sends a low-temperature chat request (temperature=0.1) for consistent JSON output.
- **`call_judge(judge, review_text, aspect, sentiment, model)`** - wraps `call_ollama` with JSON parsing and 3-attempt retry logic.

In [4]:
print(" Starting: Defining Ollama API functions...")
_start_time = time.time()
import time as t_lib

def check_ollama(model: str) -> str:
    """Check Ollama is running, model exists, and determine hardware (GPU/CPU).
    Returns the resolved full model tag (e.g. 'llama3.1:latest')."""
    try:
        r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
        models = [m["name"] for m in r.json().get("models", [])]
    except requests.exceptions.ConnectionError:
        print("\n[ERROR] Cannot connect to Ollama at http://localhost:11434")
        raise RuntimeError("Ollama not reachable")

    matched = [m for m in models if model.split(":")[0] in m]
    if not matched:
        print(f"\n[ERROR] Model '{model}' not found locally. Run: ollama pull {model}")
        raise RuntimeError(f"Model '{model}' not available")

    hw_info = "[Detecting...]"
    for attempt in range(3):
        try:
            requests.post(
                f"{OLLAMA_BASE}/api/generate",
                json={"model": matched[0], "prompt": "hi", "stream": False},
                timeout=10,
            )
            r_ps = requests.get(f"{OLLAMA_BASE}/api/ps", timeout=5)
            ps_data = r_ps.json().get("models", [])
            model_ps = next((m for m in ps_data if matched[0] in m["name"]), None)
            if model_ps:
                proc = model_ps.get("processor", "Unknown")
                if proc == "Unknown":
                    t_lib.sleep(2)
                    continue
                hw_info = f"Running on {proc}"
                break
            else:
                hw_info = "Hardware: Ollama will choose best available"
                t_lib.sleep(1)
        except Exception:
            hw_info = "Hardware: Native (check nvidia-smi for VRAM usage)"

    print(f"  {matched[0]:30s} — {hw_info}")
    return matched[0]


def check_all_models(judge_models: list) -> dict:
    """Check all judge models are available. Returns {model_name: resolved_tag}."""
    print("  Checking judge models:")
    resolved = {}
    for m in judge_models:
        tag = check_ollama(m)
        resolved[m] = tag
    return resolved


def call_ollama(system_prompt: str, user_prompt: str, model: str) -> str:
    """Send a chat request to local Ollama and return the response text."""
    payload = {
        "model":  model,
        "stream": False,
        "options": {"temperature": 0.1, "top_p": 0.9},
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
    }
    response = requests.post(f"{OLLAMA_BASE}/api/chat", json=payload, timeout=120)
    response.raise_for_status()
    return response.json()["message"]["content"]


def call_judge(judge: dict, review_text: str, aspect: str,
               sentiment: str, model: str) -> dict | None:
    """Call one judge persona on a specific model. Retries 3×."""
    prompt = JUDGE_PROMPT.format(
        review_text=review_text[:500],
        aspect=aspect,
        aspect_description=ASPECT_DESCRIPTIONS.get(aspect, aspect),
        sentiment=sentiment,
        sentiment_description=SENTIMENT_DESCRIPTIONS.get(sentiment, sentiment),
    )
    for attempt in range(3):
        try:
            raw = call_ollama(judge["system"], prompt, model).strip()
            raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.MULTILINE)
            raw = re.sub(r"\s*```$",          "", raw, flags=re.MULTILINE)
            match = re.search(r"\{.*\}", raw, re.DOTALL)
            if match:
                raw = match.group(0)
            return json.loads(raw)
        except json.JSONDecodeError:
            t_lib.sleep(1)
        except Exception:
            t_lib.sleep(2)
    return None


print(f"\n Done in {time.time() - _start_time:.2f}s")
print(" Completed: Defining Ollama API functions.")

Starting: Defining Ollama API functions...

Done in 0.00s
Completed: Defining Ollama API functions.


## 5. Statistics - Fleiss' Kappa

**Fleiss' Kappa (κ)** measures inter-rater agreement for categorical ratings across multiple raters.

Here it is applied as a **binary agreement** measure (PASS=1 / FAIL=0) across the 3 judge personas:

$$\kappa = \frac{\bar{P} - P_e}{1 - P_e}$$

| κ range | Interpretation |
|---------|----------------|
| < 0.20 | Slight |
| 0.20–0.40 | Fair |
| 0.40–0.60 | Moderate |
| 0.60–0.80 | Substantial |
| > 0.80 | Almost Perfect |


In [5]:
print(" Starting: Defining Fleiss' Kappa functions...")
_start_time = time.time()

def fleiss_kappa(ratings_matrix: np.ndarray) -> float:
    """
    Fleiss' Kappa for multi-rater binary agreement.
    """
    n_items, n_categories = ratings_matrix.shape
    n_raters = int(ratings_matrix[0].sum())
    if n_raters < 2:
        return float("nan")

    p_j   = ratings_matrix.sum(axis=0) / (n_items * n_raters)
    P_e   = (p_j ** 2).sum()
    P_i   = ((ratings_matrix ** 2).sum(axis=1) - n_raters) / (n_raters * (n_raters - 1))
    P_bar = P_i.mean()

    if abs(1 - P_e) < 1e-9:
        return 1.0
    return float((P_bar - P_e) / (1 - P_e))


def interpret_kappa(k: float) -> str:
    """Return a human-readable label for a Fleiss' Kappa value."""
    if np.isnan(k): return "N/A"
    if k < 0:       return "Poor"
    if k < 0.20:    return "Slight"
    if k < 0.40:    return "Fair"
    if k < 0.60:    return "Moderate"
    if k < 0.80:    return "Substantial"
    return "Almost Perfect"




def pabak(ratings_matrix: np.ndarray) -> float:
    """
    Prevalence-Adjusted Bias-Adjusted Kappa (PABAK).

    Corrects for the prevalence paradox: Fleiss' kappa collapses toward 0
    when pass rates approach 100%, even if raters agree almost perfectly.
    PABAK replaces the chance-agreement term with a fixed 0.5 baseline.

    Formula : PABAK = 2 * P_o - 1
    where P_o is the mean observed pairwise-agreement proportion across items.

    Reference: Byrt, Bishop & Carlin (1993), J. Clinical Epidemiology 46(5).
    """
    n_items, _ = ratings_matrix.shape
    n_raters = int(ratings_matrix[0].sum())
    if n_raters < 2:
        return float("nan")

    P_i = []
    for row in ratings_matrix:
        r = float(row[0])   # PASS votes
        agree = r * (r - 1) + (n_raters - r) * (n_raters - r - 1)
        total_pairs = n_raters * (n_raters - 1)
        P_i.append(agree / total_pairs)

    P_o = float(np.mean(P_i))
    return float(2 * P_o - 1)

print(f"\n Done in {time.time() - _start_time:.2f}s")
print(" Completed: Defining Fleiss' Kappa and PABAK functions.")

 Starting: Defining Fleiss' Kappa functions...

 Done in 0.00s
 Completed: Defining Fleiss' Kappa functions.


## 6. Step 1 - Check All Models & Load Samples

- Verifies all models in `JUDGE_MODELS` are available in Ollama.
- Loads `SAMPLE_N` random reviews from each synthetic CSV.

In [6]:
print(" Starting: Checking models and loading samples...")
_start_time = time.time()

random.seed(42)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Resolve all judge model tags (fails fast if any model is not pulled)
RESOLVED_MODELS = check_all_models(JUDGE_MODELS)
print(f"\n  Resolved models: {RESOLVED_MODELS}")

def load_samples(data_dir: Path, sample_n: int) -> list:
    samples = []
    for filename, (aspect, sentiment) in SYNTHETIC_FILES.items():
        filepath = data_dir / filename
        if not filepath.exists():
            print(f"  [WARN] Not found: {filepath}")
            continue
        df = pd.read_csv(filepath)
        text_col = next(
            (c for c in ["review", "text", "Review", "Text", "review_text", "content"]
             if c in df.columns),
            df.columns[0],
        )
        texts   = df[text_col].dropna().astype(str).tolist()
        n       = min(sample_n, len(texts))
        sampled = random.sample(texts, n)
        for t in sampled:
            samples.append({"file": filename, "aspect": aspect, "sentiment": sentiment, "review_text": t})
        print(f"  Loaded {n}/{len(texts)} samples from {filename}")
    return samples

samples = load_samples(DATA_DIR, SAMPLE_N)
total   = len(samples)
total_calls = total * len(JUDGE_MODELS) * len(JUDGE_PERSONAS)
print(f"\n  Total samples       : {total}")
print(f"  Judge calls expected: {total_calls}  ({total} × {len(JUDGE_MODELS)} models × {len(JUDGE_PERSONAS)} personas)")
print(f"\n Done in {time.time() - _start_time:.2f}s")
print(" Completed: Checking models and loading samples.")

 Starting: Checking models and loading samples...
  Checking judge models:
   llama3.1:latest                — [Detecting...]
   mistral:latest                 — [Detecting...]
   gemma2:latest                  — Hardware: Native (check nvidia-smi for VRAM usage)

  Resolved models: {'llama3.1': 'llama3.1:latest', 'mistral': 'mistral:latest', 'gemma2': 'gemma2:latest'}
  Loaded 30/192 samples from LLM_Gen_Packing_Neg_Reviews.csv
  Loaded 30/196 samples from LLM_Gen_Packing_Neu_Reviews.csv
  Loaded 30/176 samples from LLM_Gen_Price_Neg_Reviews.csv
  Loaded 30/193 samples from LLM_Gen_Price_Neu_Reviews.csv
  Loaded 30/166 samples from LLM_Gen_Smell_Neu_Reviews.csv

  Total samples       : 150
  Judge calls expected: 1350  (150 × 3 models × 3 personas)

 Done in 94.37s
 Completed: Checking models and loading samples.


## 7. Step 2 - Run Multi-Model Judge Validation

For each review:
1. **Per model**: run all 3 judge personas → per-model majority vote (≥ 2/3 personas pass)
2. **Cross-model**: average verdict across all models → final verdict (≥ 2/3 models pass)
3. **Overall valid**: all 3 criteria pass in the final verdict

Column naming: `{model}_{judge}_{criterion}` → `{model}_majority_{criterion}` → `final_{criterion}`

In [ ]:
print(f" Starting: Running {total} samples × {len(JUDGE_MODELS)} models × {len(JUDGE_PERSONAS)} personas...\n")
_start_eval_time = time.time()
results = []

pbar = tqdm(samples, desc="Validating Samples", unit="sample")
for i, sample in enumerate(pbar):
    print(f"\nSample [{i+1}/{total}] | {sample['aspect']:10s} | {sample['sentiment']:8s} | "
          f"{sample['review_text'][:50]}...")

    row = {
        "sample_id":   i + 1,
        "file":        sample["file"],
        "aspect":      sample["aspect"],
        "sentiment":   sample["sentiment"],
        "review_text": sample["review_text"],
    }

    # model_short_name → {criterion → list of persona scores}
    model_persona_scores: dict[str, dict[str, list[int]]] = {}

    for model_name, model_tag in RESOLVED_MODELS.items():
        model_key = model_name.split(":")[0]   # e.g. "llama3.1"
        model_persona_scores[model_key] = {c: [] for c in CRITERIA}

        print(f"  ┌─ Model: {model_key}")
        for judge in JUDGE_PERSONAS:
            verdict = call_judge(judge, sample["review_text"],
                                 sample["aspect"], sample["sentiment"], model_tag)

            col_prefix = f"{model_key}_{judge['name']}"
            if verdict is None:
                for c in CRITERIA:
                    row[f"{col_prefix}_{c}"] = "ERROR"
                    row[f"{col_prefix}_{c}_reason"] = ""
                print(f"  │  └─ {judge['name']:7s}: [ERROR — all criteria]")
                continue

            parsed = {}
            for c in CRITERIA:
                v = verdict.get(c, {}).get("verdict", "FAIL")
                score = 1 if str(v).upper() == "PASS" else 0
                parsed[c] = score
                model_persona_scores[model_key][c].append(score)
                row[f"{col_prefix}_{c}"] = str(v).upper()
                row[f"{col_prefix}_{c}_reason"] = verdict.get(c, {}).get("reason", "")

            icons = " ".join(f"[{'✓' if parsed[c] else '✗'} {c[0].upper()}]" for c in CRITERIA)
            print(f"  │  └─ {judge['name']:7s}: {icons}")

        # Per-model majority vote (≥ 2/3 personas)
        for c in CRITERIA:
            votes = model_persona_scores[model_key][c]
            majority = (1 if sum(votes) >= 2 else 0) if len(votes) >= 2 else 0
            row[f"{model_key}_majority_{c}"] = majority
        model_summary = "  ".join(
            f"{c[0].upper()}={'✓' if row[f'{model_key}_majority_{c}'] else '✗'}"
            for c in CRITERIA
        )
        print(f"  └─ {model_key} majority: {model_summary}")

    # Cross-model final verdict: criterion passes if ≥ ceil(n_models/2)+1 models agree
    # (i.e. majority of models pass that criterion)
    n_models = len(RESOLVED_MODELS)
    pass_threshold = (n_models // 2) + 1   # e.g. 2 out of 3
    for c in CRITERIA:
        model_votes = [row.get(f"{mk}_majority_{c}", 0) for mk in model_persona_scores]
        row[f"final_{c}"] = 1 if sum(model_votes) >= pass_threshold else 0
        # Average PASS rate across models (0.0–1.0) for reporting
        row[f"avg_score_{c}"] = round(sum(model_votes) / n_models, 3) if n_models else 0

    row["overall_valid"] = int(all(row.get(f"final_{c}", 0) == 1 for c in CRITERIA))
    results.append(row)

total_duration = time.time() - _start_eval_time
print(f"\n Total Evaluation Time: {timedelta(seconds=int(total_duration))}")
print(" Completed: Multi-model judge validation.")

 Starting: Running 150 samples × 3 models × 3 personas...



Validating Samples:   0%|          | 0/150 [00:00<?, ?sample/s]


Sample [1/150] | packing    | negative | Packaging was not secure, the box opened easily du...
  ┌─ Model: llama3.1
  │  └─ Judge_A: [✓ L] [✓ A] [✓ N]
  │  └─ Judge_B: [✓ L] [✗ A] [✓ N]
  │  └─ Judge_C: [✓ L] [✓ A] [✓ N]
  └─ llama3.1 majority: L=✓  A=✓  N=✓
  ┌─ Model: mistral
  │  └─ Judge_A: [✓ L] [✓ A] [✓ N]
  │  └─ Judge_B: [✓ L] [✓ A] [✓ N]
  │  └─ Judge_C: [✓ L] [✓ A] [✓ N]
  └─ mistral majority: L=✓  A=✓  N=✓
  ┌─ Model: gemma2
  │  └─ Judge_A: [✓ L] [✓ A] [✓ N]
  │  └─ Judge_B: [✓ L] [✓ A] [✓ N]
  │  └─ Judge_C: [✓ L] [✓ A] [✓ N]
  └─ gemma2 majority: L=✓  A=✓  N=✓

Sample [2/150] | packing    | negative | Bad packaging, the box was already dented before o...
  ┌─ Model: llama3.1
  │  └─ Judge_A: [✓ L] [✓ A] [✓ N]
  │  └─ Judge_B: [✓ L] [✓ A] [✗ N]
  │  └─ Judge_C: [✓ L] [✓ A] [✓ N]
  └─ llama3.1 majority: L=✓  A=✓  N=✓
  ┌─ Model: mistral
  │  └─ Judge_A: [✓ L] [✓ A] [✓ N]
  │  └─ Judge_B: [✓ L] [✓ A] [✓ N]
  │  └─ Judge_C: [✓ L] [✓ A] [✓ N]
  └─ mistral majority: L=✓  A=✓  

## 8. Step 3 - Compute Statistics & Save CSV

In [8]:
print(" Starting: Computing statistics and saving results CSV...")
df_results = pd.DataFrame(results)
csv_path = OUTPUT_DIR / "validation_results.csv"
df_results.to_csv(csv_path, index=False)
print(f"   Saved raw results → {csv_path}")

n_total = len(df_results)
if n_total == 0:
    print(" No results found.")
    n_valid = 0; pct = 0
else:
    n_valid = int(df_results["overall_valid"].sum())
    pct = 100 * n_valid / n_total

    # Final criterion pass rates (cross-model majority)
    criterion_rates = {c: 100 * df_results[f"final_{c}"].mean() for c in CRITERIA}

    # Average score per criterion across models
    avg_scores = {c: df_results[f"avg_score_{c}"].mean() for c in CRITERIA}

    # Per-model pass rates
    model_keys = list(RESOLVED_MODELS.keys())
    per_model_rates = {}
    for mk in model_keys:
        per_model_rates[mk] = {
            c: 100 * df_results[f"{mk}_majority_{c}"].mean() for c in CRITERIA
        }

    per_file   = df_results.groupby("file")["overall_valid"].agg(["sum", "count"])
    per_file["pct"] = 100 * per_file["sum"] / per_file["count"]
    per_aspect = df_results.groupby("aspect")["overall_valid"].agg(["sum", "count"])
    per_aspect["pct"] = 100 * per_aspect["sum"] / per_aspect["count"]

    # Fleiss' Kappa — treat every model-persona combination as an independent rater
    # Total raters = len(JUDGE_MODELS) × len(JUDGE_PERSONAS)
    kappas = {}
    all_judge_cols = [
        f"{mk}_{j['name']}"
        for mk in model_keys
        for j in JUDGE_PERSONAS
    ]
    for c in CRITERIA:
        ratings = []
        for _, r in df_results.iterrows():
            passes = sum(
                1 for col in all_judge_cols
                if str(r.get(f"{col}_{c}", "FAIL")).upper() == "PASS"
            )
            n_raters = len(all_judge_cols)
            ratings.append([passes, n_raters - passes])
        kappas[c] = fleiss_kappa(np.array(ratings))


    # PABAK — Prevalence-Adjusted Bias-Adjusted Kappa
    # Corrects Fleiss' kappa for the prevalence paradox (kappa -> 0 at ~100% pass rate).
    pabaks = {}
    for c in CRITERIA:
        ratings = []
        for _, r in df_results.iterrows():
            passes = sum(
                1 for col in all_judge_cols
                if str(r.get(f"{col}_{c}", "FAIL")).upper() == "PASS"
            )
            n_raters = len(all_judge_cols)
            ratings.append([passes, n_raters - passes])
        pabaks[c] = pabak(np.array(ratings))

print(" Completed: Computing statistics.")

 Starting: Computing statistics and saving results CSV...
   Saved raw results → ..\data\augmented\validation_output\validation_results.csv
 Completed: Computing statistics.


## 9. Results Summary

In [ ]:
print(f"\n{'='*70}\nMULTI-MODEL LLM-AS-A-JUDGE RESULTS\n{'='*70}")
print(f"  Models consulted  : {', '.join(model_keys)}")
print(f"  Judges per review : {len(model_keys)} models × {len(JUDGE_PERSONAS)} personas = {len(model_keys)*len(JUDGE_PERSONAS)}")
print(f"  Samples evaluated : {n_total}")
print(f"  Overall valid     : {n_valid}/{n_total} ({pct:.1f}%)\n")

print("  ── Final (cross-model average) ──────────────────────────────────────")
for c in CRITERIA:
    k  = kappas.get(c, 0)
    pk = pabaks.get(c, 0)
    print(f"  {c.replace('_',' ').title():22s}: {criterion_rates.get(c,0):5.1f}%  "
          f"avg={avg_scores.get(c,0):.2f}  "
          f"kappa={k:.3f} ({interpret_kappa(k)})  "
          f"PABAK={pk:.3f} ({interpret_kappa(pk)})")
print()
print("  NOTE: Fleiss kappa is low due to the prevalence paradox (~99% pass rate).")
print("        PABAK corrects for this and is the appropriate metric here.")
print("        Ref: Byrt, Bishop & Carlin (1993), J. Clin. Epidemiology.")
print(f"  {c.replace('_',' ').title():22s}: {criterion_rates.get(c,0):5.1f}%  "
          f"avg={avg_scores.get(c,0):.2f}  κ={kappas.get(c,0):.3f} ({interpret_kappa(kappas.get(c,0))})")

print("\n  ── Per-model breakdown ──────────────────────────────────────────────")
header = f"  {'Criterion':22s}" + "".join(f"  {mk:>12s}" for mk in model_keys)
print(header)
for c in CRITERIA:
    row_str = f"  {c.replace('_',' ').title():22s}"
    for mk in model_keys:
        row_str += f"  {per_model_rates[mk][c]:>10.1f}%"
    print(row_str)

print(f"\n  ── Per-file ──────────────────────────────────────────────────────────")
for fn, r in per_file.iterrows():
    print(f"    {fn:45s}: {r['pct']:.1f}%")
print(f"{'='*70}\n")


MULTI-MODEL LLM-AS-A-JUDGE RESULTS
  Models consulted  : llama3.1, mistral, gemma2
  Judges per review : 3 models × 3 personas = 9
  Samples evaluated : 150
  Overall valid     : 149/150 (99.3%)

  ── Final (cross-model average) ──────────────────────────────────────
  Label Correctness     : 100.0%  avg=1.00  κ=-0.009 (Poor)
  Aspect Relevance      : 100.0%  avg=0.94  κ=0.092 (Slight)
  Naturalness           :  99.3%  avg=0.97  κ=0.096 (Slight)

  ── Per-model breakdown ──────────────────────────────────────────────
  Criterion                   llama3.1       mistral        gemma2
  Label Correctness            100.0%       100.0%       100.0%
  Aspect Relevance              82.7%       100.0%       100.0%
  Naturalness                   90.7%        99.3%       100.0%

  ── Per-file ──────────────────────────────────────────────────────────
    LLM_Gen_Packing_Neg_Reviews.csv              : 100.0%
    LLM_Gen_Packing_Neu_Reviews.csv              : 100.0%
    LLM_Gen_Price_Neg_Revie

## 10. Interactive Exploration

In [10]:
if n_total > 0:
    # Sample-level overview
    overview_cols = ["sample_id", "aspect", "sentiment", "overall_valid"] + \
                    [f"final_{c}" for c in CRITERIA] + \
                    [f"avg_score_{c}" for c in CRITERIA]
    display(df_results[overview_cols].head(10))

    # Kappa summary
    display(pd.DataFrame([{
        "Criterion": c.replace("_", " ").title(),
        "Final Pass%": f"{criterion_rates[c]:.1f}%",
        "Avg Score": f"{avg_scores[c]:.2f}",
        "Fleiss kappa (9 raters)": f"{kappas[c]:.3f}",
        "Fleiss Agreement": interpret_kappa(kappas[c]),
        "PABAK": f"{pabaks[c]:.3f}",
        "PABAK Agreement": interpret_kappa(pabaks[c]),
    } for c in CRITERIA]))

    # Per-model pass rates
    per_model_df = pd.DataFrame([{
        "Model": mk,
        **{c.replace("_", " ").title(): f"{per_model_rates[mk][c]:.1f}%" for c in CRITERIA},
    } for mk in model_keys])
    print("\nPer-model majority pass rates:")
    display(per_model_df)

,sample_id,aspect,sentiment,overall_valid,final_label_correctness,final_aspect_relevance,final_naturalness,avg_score_label_correctness,avg_score_aspect_relevance,avg_score_naturalness
0,1,packing,negative,1,1,1,1,1.0,1.000,1.0
1,2,packing,negative,1,1,1,1,1.0,1.000,1.0
2,3,packing,negative,1,1,1,1,1.0,1.000,1.0
3,4,packing,negative,1,1,1,1,1.0,1.000,1.0
4,5,packing,negative,1,1,1,1,1.0,0.667,1.0
5,6,packing,negative,1,1,1,1,1.0,1.000,1.0
6,7,packing,negative,1,1,1,1,1.0,0.667,1.0
7,8,packing,negative,1,1,1,1,1.0,0.667,1.0
8,9,packing,negative,1,1,1,1,1.0,0.667,1.0
9,10,packing,negative,1,1,1,1,1.0,0.667,1.0


,Criterion,Final Pass%,Avg Score,Fleiss κ (9 raters),Agreement
0,Label Correctness,100.0%,1.00,-0.009,Poor
1,Aspect Relevance,100.0%,0.94,0.092,Slight
2,Naturalness,99.3%,0.97,0.096,Slight



Per-model majority pass rates:


,Model,Label Correctness,Aspect Relevance,Naturalness
0,llama3.1,100.0%,82.7%,90.7%
1,mistral,100.0%,100.0%,99.3%
2,gemma2,100.0%,100.0%,100.0%
